In [1]:
model_name = "vgg11-ragdoll"
model_file = "vgg11.mlir.v0"

import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n

image = torch.randn(1, 3, 224, 224)
image_np = image.detach().cpu().numpy()
image_transposed = torch.randn(1, 224, 224, 3)
image_np_transposed = image_transposed.detach().cpu().numpy()
model = models.vgg11().train(False)
model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
output = model(image)
grad = torch.randn_like(output)
grad_np = grad.cpu().numpy()

BENCHMARK_REPEAT=33
df = pd.DataFrame()

In [2]:
model_file_base = "vgg11.mlir.v0"
#for bs in [1, 2, 4, 8, 16]:
#for bs in [32, 64, 128, 256, 512, 1024]:
bs=1
model_file = model_file_base

target_file = model_file + ".recompute"
!ragdoll-opt {model_file}  \
--canonicalize \
--enable-cse-in-legalizer \
--symbol-dce \
--ragdoll-autodiff-vjp-public-functions='strategy=recompute' \
--ragdoll-autodiff-vjp \
--inline \
--ragdoll-autodiff-inline-function-call \
--ragdoll-initialisation \
--eliminate-empty-tensors \
--ragdoll-legalise-to-iree-compatibility \
--ragdoll-raise-linalg-to-tosa \
--canonicalize \
--cse > {target_file}

target_file = model_file + ".storeall"
!ragdoll-opt {model_file}  \
--canonicalize \
--enable-cse-in-legalizer \
--symbol-dce \
--ragdoll-autodiff-vjp-public-functions='strategy=storeall' \
--ragdoll-autodiff-vjp \
--inline \
--ragdoll-autodiff-inline-function-call \
--ragdoll-initialisation \
--eliminate-empty-tensors \
--ragdoll-legalise-to-iree-compatibility \
--ragdoll-raise-linalg-to-tosa \
--canonicalize \
--cse > {target_file}

target_file = model_file + ".heuristic"
!ragdoll-opt {model_file}  \
--canonicalize \
--enable-cse-in-legalizer \
--symbol-dce \
--ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
--ragdoll-autodiff-vjp \
--inline \
--ragdoll-autodiff-inline-function-call \
--ragdoll-initialisation \
--eliminate-empty-tensors \
--ragdoll-legalise-to-iree-compatibility \
--ragdoll-raise-linalg-to-tosa \
--ragdoll-forward-func-removal \
--canonicalize \
--cse > {target_file}
    
#!iree-compile recompute.mlir \
!iree-compile {model_file}.recompute \
-o {model_file}.recompute.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86

!iree-compile {model_file}.storeall \
-o {model_file}.storeall.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86

"""
!iree-compile {model_file}.hybrid \
-o {model_file}.hybrid.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86
"""

!iree-compile {model_file}.heuristic \
-o {model_file}.heuristic.vmfb \
--iree-hal-target-backends=cuda \
--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
--iree-hal-cuda-llvm-target-arch=sm_86

#!iree-compile {model_file}.heuristic{"-mulswap"} \
#-o {model_file}.heuristic{"-mulswap"}.vmfb \
#--iree-hal-target-backends=cuda \
#--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
#--iree-hal-cuda-llvm-target-arch=sm_86

recompute = model_file+".recompute.vmfb"
storeall = model_file+".storeall.vmfb"
#hybrid = model_file+".hybrid.vmfb"

#heuristic2 = model_file+".heuristic-mulswap.vmfb"

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

#recompute_fb, storeall_fb = [
#    load_executable(x) for x in [recompute, storeall]
#]
recompute_fb = load_executable(recompute)
storeall_fb = load_executable(storeall)
#hybrid_fb = load_executable(hybrid)
#heuristic_fb = load_executable(heuristic)
#heuristic2_fb = load_executable(heuristic2)

In [3]:
#f1 = timeit("recompute_fb.forward(image_np)") / BENCHMARK_REPEAT
f1 = timeit("recompute_fb.forward(image_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', f1)
b1 = timeit("recompute_fb.dforward(grad_np)") /  BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-backward in timeit: ', b1)
df = pd.concat([df, get_dataframe(f1, b1, "Nabla-Recompute")])
print(df)

ragdoll-opt1-gpu-forward in timeit:  1.9342754826401218
ragdoll-opt1-gpu-backward in timeit:  8.313180342542402
        time      pass             item
0   1.934275   Forward  Nabla-Recompute
0   8.313180  Backward  Nabla-Recompute
0  10.247456      Full  Nabla-Recompute


In [4]:
f1 = timeit("storeall_fb.forward(image_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', f1)
b1 = timeit("storeall_fb.dforward(grad_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', b1)
df = pd.concat([df, get_dataframe(f1, b1, "Nabla-Storeall")])
print(df)

ragdoll-opt1-gpu-forward in timeit:  2.7355821238774243
ragdoll-opt1-gpu-forward in timeit:  23.026458806160726
        time      pass             item
0   1.934275   Forward  Nabla-Recompute
0   8.313180  Backward  Nabla-Recompute
0  10.247456      Full  Nabla-Recompute
0   2.735582   Forward   Nabla-Storeall
0  23.026459  Backward   Nabla-Storeall
0  25.762041      Full   Nabla-Storeall


In [6]:
#f1 = timeit("heuristic_fb.forward(image_np_transposed)") / BENCHMARK_REPEAT
#print('ragdoll-opt1-gpu-forward in timeit: ', f1)
#b1 = timeit("heuristic_fb.dforward(grad_np)") / BENCHMARK_REPEAT
#print('ragdoll-opt1-gpu-forward in timeit: ', b1)
heuristic = model_file+".heuristic.vmfb"
b1 = ragdoll_model_benchmark(
    heuristic,
    "dforward",
    [(1, 1000)],
    device='gpu',
    warmups=2,
    repetitions=BENCHMARK_REPEAT, 
    measure_count=3)
b1 = np.mean(b1)
df = pd.concat([df, get_dataframe(b1, b1, "Nabla-Heuristic")])
print(df)

(1, 1000)
['1x1000xf32']
        time      pass             item
0   1.934275   Forward  Nabla-Recompute
0   8.313180  Backward  Nabla-Recompute
0  10.247456      Full  Nabla-Recompute
0   2.735582   Forward   Nabla-Storeall
0  23.026459  Backward   Nabla-Storeall
0  25.762041      Full   Nabla-Storeall
0   6.780000   Forward  Nabla-Heuristic
0   6.780000  Backward  Nabla-Heuristic
0  13.560000      Full  Nabla-Heuristic


In [8]:
heuristic = model_file+".heuristic.vmfb"
heuristic_fb = load_executable(heuristic)
#f1 = timeit("heuristic_fb.forward(image_np)") / BENCHMARK_REPEAT
#print('ragdoll-opt1-gpu-forward in timeit: ', f1)
b1 = timeit("heuristic_fb.dforward(grad_np)") / BENCHMARK_REPEAT
print('ragdoll-opt1-gpu-forward in timeit: ', b1)
df = pd.concat([df, get_dataframe(b1, b1, "Nabla-Heuristic-timeit")])
print(df)

ragdoll-opt1-gpu-forward in timeit:  7.2190553570787115
        time      pass                    item
0   1.934275   Forward         Nabla-Recompute
0   8.313180  Backward         Nabla-Recompute
0  10.247456      Full         Nabla-Recompute
0   2.735582   Forward          Nabla-Storeall
0  23.026459  Backward          Nabla-Storeall
0  25.762041      Full          Nabla-Storeall
0   6.780000   Forward         Nabla-Heuristic
0   6.780000  Backward         Nabla-Heuristic
0  13.560000      Full         Nabla-Heuristic
0   7.219055   Forward  Nabla-Heuristic-timeit
0   7.219055  Backward  Nabla-Heuristic-timeit
0  14.438111      Full  Nabla-Heuristic-timeit


In [ ]:
df.style.hide(axis="index")
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")